# End-to-end validation
Checks every expected Silver, dimension, and fact table, followed by atomic-grain and unresolved-key diagnostics.

In [ ]:
from pyspark.sql import functions as F, types as T
from delta.tables import DeltaTable
from functools import reduce
from datetime import datetime, timezone
import uuid

BRZ_SLV_LH = "lh_brz_slv_finance"
GLD_LH = "lh_gld_finance"
BRONZE_ROOT = "Files/bronze/sap_finance"
SLV = f"{BRZ_SLV_LH}.slv"
QTN = f"{BRZ_SLV_LH}.quarantine"
AUD = f"{BRZ_SLV_LH}.audit"
DIM = f"{GLD_LH}.dim"
FCT = f"{GLD_LH}.fct"
SOURCE_SYSTEM = "SAP_S4HANA"
BATCH_RUN_ID = str(uuid.uuid4())
LOAD_TIME_UTC = datetime.now(timezone.utc).replace(tzinfo=None)

spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.microsoft.delta.autoCompact.enabled", "true")
print(BATCH_RUN_ID, LOAD_TIME_UTC)

In [ ]:
objects = (
    [f"{SLV}.{table_name}" for table_name in [
        "acdoca", "bkpf", "t001", "csks", "cepc", "ska1", "skat",
        "lfa1", "kna1", "faglflexp", "budget",
    ]]
    + [f"{DIM}.{table_name}" for table_name in [
        "company", "gl_account", "cost_center", "profit_center",
        "division", "vendor", "customer", "date", "fiscal_period",
    ]]
    + [f"{FCT}.{table_name}" for table_name in [
        "finance_transaction", "budget",
    ]]
    + [
        f"{GLD_LH}.mlv.finance_actual_monthly",
        f"{GLD_LH}.mlv.vendor_spend_monthly",
        f"{GLD_LH}.mlv.gl_balance_monthly",
        f"{GLD_LH}.mlv.plan_vs_actual_monthly",
    ]
)

rows = []
for object_name in objects:
    try:
        rows.append((object_name, spark.table(object_name).count(), True, ""))
    except Exception as error:
        rows.append((object_name, 0, False, str(error)[:400]))

result = (
    spark.createDataFrame(
        rows,
        "object_name string, row_count long, exists boolean, error string",
    )
    .withColumn("batch_run_id", F.lit(BATCH_RUN_ID))
    .withColumn("check_time", F.lit(LOAD_TIME_UTC))
)

missing_objects = result.filter(~F.col("exists")).count()
empty_objects = result.filter(F.col("exists") & (F.col("row_count") == 0)).count()

fact = spark.table(f"{FCT}.finance_transaction")
duplicate_fact_keys = (
    fact.groupBy("FINANCE_TRANSACTION_SK")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

mandatory_dimension_keys = [
    "COMPANY_SK", "GL_ACCOUNT_SK", "COST_CENTER_SK",
    "PROFIT_CENTER_SK", "DIVISION_SK",
]
unresolved_mandatory = {
    key_name: fact.filter(F.col(key_name).isNull()).count()
    for key_name in mandatory_dimension_keys
}
unresolved_optional = {
    "VENDOR_SK": fact.filter(
        F.col("LIFNR").isNotNull() & F.col("VENDOR_SK").isNull()
    ).count(),
    "CUSTOMER_SK": fact.filter(
        F.col("KUNNR").isNotNull() & F.col("CUSTOMER_SK").isNull()
    ).count(),
}

latest_acdoca_extract = spark.table(f"{SLV}.acdoca").agg(
    F.max("EXTRACT_DATE").alias("EXTRACT_DATE")
).first()["EXTRACT_DATE"]
expected_fact_rows = (
    spark.table(f"{SLV}.acdoca")
    .filter(F.col("EXTRACT_DATE") == F.lit(latest_acdoca_extract))
    .select("RCLNT", "RLDNR", "RBUKRS", "GJAHR", "BELNR", "DOCLN")
    .dropDuplicates()
    .count()
)
actual_fact_rows = fact.select("FINANCE_TRANSACTION_SK").distinct().count()

fiscal_period_dimension = spark.table(f"{DIM}.fiscal_period")
duplicate_fiscal_period_keys = (
    fiscal_period_dimension.groupBy("FISCAL_PERIOD_KEY")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
fiscal_period_sources = [
    f"{FCT}.finance_transaction",
    f"{FCT}.budget",
    f"{GLD_LH}.mlv.finance_actual_monthly",
    f"{GLD_LH}.mlv.vendor_spend_monthly",
    f"{GLD_LH}.mlv.gl_balance_monthly",
    f"{GLD_LH}.mlv.plan_vs_actual_monthly",
]
unresolved_fiscal_period_keys = 0
for source_name in fiscal_period_sources:
    source_keys = spark.table(source_name).select(
        F.col("FISCAL_PERIOD_KEY").cast("int").alias("FISCAL_PERIOD_KEY")
    )
    unresolved_fiscal_period_keys += (
        source_keys.filter(F.col("FISCAL_PERIOD_KEY").isNull()).count()
        + source_keys.filter(F.col("FISCAL_PERIOD_KEY").isNotNull())
        .join(
            fiscal_period_dimension.select("FISCAL_PERIOD_KEY"),
            on="FISCAL_PERIOD_KEY",
            how="left_anti",
        )
        .count()
    )

budget_fact = spark.table(f"{FCT}.budget")
duplicate_budget_keys = (
    budget_fact.groupBy("BUDGET_FACT_SK")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

checks = [
    ("required_objects_exist", missing_objects == 0, missing_objects),
    ("required_objects_non_empty", empty_objects == 0, empty_objects),
    ("finance_fact_key_unique", duplicate_fact_keys == 0, duplicate_fact_keys),
    (
        "finance_fact_reconciles_to_latest_acdoca",
        expected_fact_rows == actual_fact_rows,
        actual_fact_rows - expected_fact_rows,
    ),
    (
        "mandatory_dimension_keys_resolved",
        sum(unresolved_mandatory.values()) == 0,
        sum(unresolved_mandatory.values()),
    ),
    (
        "populated_optional_dimension_keys_resolved",
        sum(unresolved_optional.values()) == 0,
        sum(unresolved_optional.values()),
    ),
    ("budget_fact_key_unique", duplicate_budget_keys == 0, duplicate_budget_keys),
    (
        "fiscal_period_dimension_key_unique",
        duplicate_fiscal_period_keys == 0,
        duplicate_fiscal_period_keys,
    ),
    (
        "fiscal_period_keys_resolved",
        unresolved_fiscal_period_keys == 0,
        unresolved_fiscal_period_keys,
    ),
]
check_result = spark.createDataFrame(
    checks,
    "check_name string, passed boolean, failure_count long",
)

result.write.mode("append").format("delta").saveAsTable(
    f"{AUD}.end_to_end_validation"
)
display(result.orderBy("object_name"))
display(check_result.orderBy("check_name"))

failed_checks = check_result.filter(~F.col("passed")).collect()
if failed_checks:
    details = ", ".join(
        f"{row['check_name']}={row['failure_count']}" for row in failed_checks
    )
    raise RuntimeError(f"End-to-end validation failed: {details}")
